# Resampling of REPLACE-BG

In [158]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

### Read Existing Tables

In [2]:
file_path = "../../data/out/REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5/"
file_map = {
    'cgm': 'ReplaceBG_cgm_history.csv.gz',
    'bolus': 'ReplaceBG_bolus_event_history.csv.gz',
    'basal': 'ReplaceBG_basal_event_history.csv.gz',
}

In [120]:
def get_df_from_file(file_path, file_name, parse_datetime=True, sep=','):
    df = pd.read_csv(file_path + file_name, sep=sep)
    if parse_datetime:
        df['datetime'] = pd.to_datetime(df['datetime'], unit='s')
    return df

In [72]:
def get_extended_df(original_df, value_column):
    """
    Get a df where quantities are distributed throughout 5-minute intervals instead of having start- and end dates.
    """
    new_rows = []
    for _, row in original_df.iterrows():
        new_rows.extend(split_duration(row, value_column))
    extended_df = pd.DataFrame(new_rows)
    extended_df.set_index('datetime', inplace=True)
    return extended_df

def split_duration(row, value_column):
    """
    For features with a duration, we split the values across 5-minute intervals by adding
    new rows for every 5-minute window in duration, and equally split the original quantity across those rows.
    """
    duration = row['end_date'] - row['datetime']
    rounded_duration = round(duration / pd.Timedelta(minutes=5)) * pd.Timedelta(minutes=5)
    num_intervals = rounded_duration // pd.Timedelta(minutes=5)
    if num_intervals < 1:
        num_intervals = 1
    value_per_interval = row[value_column] / num_intervals
    new_rows = []
    for i in range(int(num_intervals)):
        new_row = {
            'datetime': row['datetime'] + pd.Timedelta(minutes=5 * i),
            value_column: value_per_interval,
            'patient_id': row['patient_id'],
        }
        new_rows.append(new_row)
    return new_rows

In [97]:
df_glucose = get_df_from_file(file_path, file_map['cgm'])
df_bolus = get_df_from_file(file_path, file_map['bolus'])
df_basal = get_df_from_file(file_path, file_map['basal'])


### Resample Existing Tables

In [98]:
df_glucose.set_index('datetime', inplace=True)
df_glucose.head()

,patient_id,cgm
datetime,,
2014-12-01 21:22:53,10,86
2014-12-01 21:27:53,10,91
2014-12-01 21:32:53,10,94
2014-12-01 21:37:53,10,99
2014-12-01 21:42:53,10,102


In [99]:
df_bolus_orig = df_bolus.copy()
df_bolus['end_date'] = df_bolus['datetime'] + pd.to_timedelta(df_bolus['delivery_duration'], unit='s')
df_bolus = get_extended_df(df_bolus, 'bolus')
df_bolus

,bolus,patient_id
datetime,,
2013-09-28 07:05:43,3.160000,10
2013-09-28 12:05:45,3.330000,10
2013-09-28 13:28:20,1.670000,10
2013-09-28 17:23:16,2.920000,10
2013-09-29 07:11:45,4.740000,10
...,...,...
2015-09-16 00:28:22,0.183333,98
2015-09-16 00:33:22,0.183333,98
2015-09-16 00:38:22,0.183333,98


In [100]:
print(f'New sum after distribution of extended boluses: {df_bolus["bolus"].sum()}, should be: {df_bolus_orig["bolus"].sum():.2f}')

New sum after distribution of extended boluses: 1457583.57, should be: 1457583.57


In [107]:
df_basal_orig = df_basal.copy()
df_basal.sort_values(by=['patient_id', 'datetime'], inplace=True)
df_basal.set_index('datetime', inplace=True)
df_basal

,patient_id,basal_rate
datetime,,
2014-08-31 21:02:33,2,0.900
2014-08-31 21:07:34,2,0.108
2014-09-01 00:00:00,2,0.108
2014-09-02 00:00:00,2,0.108
2014-09-03 00:00:00,2,0.108
...,...,...
2015-07-11 21:47:53,293,0.475
2015-07-12 00:00:00,293,0.400
2015-07-12 03:00:00,293,0.550


In [135]:
processed_dfs = []
subject_ids = df_glucose['patient_id'].unique()
print("Subjects:", len(subject_ids))
for subject_id in subject_ids:
    df_subject = df_glucose[df_glucose['patient_id'] == subject_id].copy()
    df_subject = df_subject[['cgm']].resample('5min', label='right').mean()
    df_subject['patient_id'] = subject_id
    df_subject.sort_index(inplace=True)

    def merge_data(df_col, df_subject, col_names, subject_id, agg_type='sum'):
        """ agg_type is data aggregation type. """
        df_subset = df_col[df_col['patient_id'] == subject_id].copy()
        if not df_subset.empty:
            if agg_type == 'mean':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').mean()
            elif agg_type == 'sum':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').sum()
            elif agg_type == 'first':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').first()
            elif agg_type == 'last':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').last()
            else:
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').sum()
            df_subject = pd.merge(df_subject, df_subset, on="datetime", how='outer')
        else:
            df_subject[col_names] = np.nan
        return df_subject

    # Add insulin and insulin type
    df_subject = merge_data(df_bolus, df_subject, ['bolus'], subject_id, agg_type='sum')
    df_subject = merge_data(df_basal, df_subject, ['basal_rate'], subject_id, agg_type='last')
    df_subject['basal_rate'] = df_subject['basal_rate'].ffill()
    
    df_subject['patient_id'] = subject_id
    df_subject = df_subject.rename(columns={'patient_id': 'id', 'basal_rate': 'basal', 'cgm': 'CGM'})
    processed_dfs.append(df_subject)
    print(f"{subject_id} is finished processing")

df_final = pd.concat(processed_dfs)

Subjects: 208
10 is finished processing
101 is finished processing
102 is finished processing
103 is finished processing
105 is finished processing
106 is finished processing
108 is finished processing
109 is finished processing
11 is finished processing
110 is finished processing
111 is finished processing
112 is finished processing
115 is finished processing
116 is finished processing
118 is finished processing
119 is finished processing
121 is finished processing
123 is finished processing
124 is finished processing
127 is finished processing
128 is finished processing
129 is finished processing
130 is finished processing
131 is finished processing
132 is finished processing
134 is finished processing
135 is finished processing
136 is finished processing
137 is finished processing
138 is finished processing
139 is finished processing
14 is finished processing
140 is finished processing
141 is finished processing
143 is finished processing
145 is finished processing
146 is finished p

In [137]:
df_final

,CGM,id,bolus,basal
datetime,,,,
2013-09-28 03:35:00,NaN,10,NaN,0.65
2013-09-28 03:40:00,NaN,10,NaN,0.65
2013-09-28 03:45:00,NaN,10,NaN,0.65
2013-09-28 03:50:00,NaN,10,NaN,0.65
2013-09-28 03:55:00,NaN,10,NaN,0.65
...,...,...,...,...
2015-09-18 12:05:00,106.0,98,NaN,1.00
2015-09-18 12:10:00,104.0,98,NaN,1.00
2015-09-18 12:15:00,100.0,98,NaN,1.00


### Add Additional Tables

We found:
- Carbs
- Insulin type
- Age
- Weight
- Height
- Gender

In [112]:
raw_data_file_path = "../../data/raw/REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5/Data Tables/"

In [128]:
#imaginary start date we chose since data is relative to enrollment (consistent with processing script)
enrollment_start = datetime(2015, 1, 1)

In [133]:
# Adding carbs
df_carbs = get_df_from_file(raw_data_file_path, 'HDeviceWizard.txt', parse_datetime=False, sep='|')
df_carbs['datetime'] = enrollment_start + pd.to_timedelta(df_carbs['DeviceDtTmDaysFromEnroll'], unit='D') + pd.to_timedelta(df_carbs['DeviceTm'])
df_carbs = df_carbs[df_carbs['CarbInput'] > 0][['PtId', 'datetime', 'CarbInput']]
df_carbs = df_carbs.rename(columns={'PtId': 'id', 'CarbInput': 'carbs'})
df_carbs.head()

,id,datetime,carbs
0,263,2014-11-09 17:45:16,25.0
1,263,2014-12-01 19:46:43,20.0
2,263,2014-12-06 18:23:24,45.0
3,263,2014-12-02 09:00:26,10.0
4,263,2014-11-02 14:26:53,30.0


In [139]:
# Add carbs to df final
processed_dfs = []
subject_ids = df_final['id'].unique()
print("Subjects:", len(subject_ids))
for subject_id in subject_ids:
    df_subject = df_final[df_final['id'] == subject_id].copy()
    df_subject.sort_index(inplace=True)

    # Add carbs
    df_subject_carbs = df_carbs[df_carbs['id'] == subject_id].copy()
    df_subject_carbs.set_index('datetime', inplace=True)
    df_subject_carbs = df_subject_carbs[df_subject_carbs['carbs'].notna()]['carbs'].resample('5min', label='right').sum()
    df_subject = pd.merge(df_subject, df_subject_carbs, on="datetime", how='outer')
    
    df_subject['id'] = subject_id
    processed_dfs.append(df_subject)
    print(f"{subject_id} is finished processing")

df_final = pd.concat(processed_dfs)
df_final

Subjects: 208
10 is finished processing
101 is finished processing
102 is finished processing
103 is finished processing
105 is finished processing
106 is finished processing
108 is finished processing
109 is finished processing
11 is finished processing
110 is finished processing
111 is finished processing
112 is finished processing
115 is finished processing
116 is finished processing
118 is finished processing
119 is finished processing
121 is finished processing
123 is finished processing
124 is finished processing
127 is finished processing
128 is finished processing
129 is finished processing
130 is finished processing
131 is finished processing
132 is finished processing
134 is finished processing
135 is finished processing
136 is finished processing
137 is finished processing
138 is finished processing
139 is finished processing
14 is finished processing
140 is finished processing
141 is finished processing
143 is finished processing
145 is finished processing
146 is finished p

,CGM,id,bolus,basal,carbs
datetime,,,,,
2013-09-28 03:35:00,NaN,10,NaN,0.65,NaN
2013-09-28 03:40:00,NaN,10,NaN,0.65,NaN
2013-09-28 03:45:00,NaN,10,NaN,0.65,NaN
2013-09-28 03:50:00,NaN,10,NaN,0.65,NaN
2013-09-28 03:55:00,NaN,10,NaN,0.65,NaN
...,...,...,...,...,...
2015-09-18 12:05:00,106.0,98,NaN,1.00,NaN
2015-09-18 12:10:00,104.0,98,NaN,1.00,NaN
2015-09-18 12:15:00,100.0,98,NaN,1.00,NaN


In [147]:
df_insulin_type = get_df_from_file(raw_data_file_path, 'HInsulin.txt', parse_datetime=False, sep='|')
df_insulin_type = df_insulin_type[['PtID', 'InsName']]
df_insulin_type = df_insulin_type.rename(columns={'PtID': 'id', 'InsName': 'insulin_type'})
df_insulin_type

,id,insulin_type
0,40,Novolog (Aspart)
1,70,Humalog (Lispro)
2,293,Novolog (Aspart)
3,137,Novolog (Aspart)
4,143,Novolog (Aspart)
...,...,...
242,271,Novolog (Aspart)
243,124,Novolog (Aspart)
244,215,Novolog (Aspart)
245,233,Novolog (Aspart)


In [148]:
def add_single_value_to_subjects(df, df_new_val, col_name):
    # TODO: this function is very inefficient... add value directly to located rows instead
    processed_dfs = []
    subject_ids = df['id'].unique()
    for subject_id in subject_ids:
        df_subject = df[df['id'] == subject_id].copy()
        df_subject.sort_index(inplace=True)
    
        user_data = df_new_val[df_new_val['id'] == subject_id].copy()
        if not user_data.empty:
            df_subject[col_name] = user_data[col_name].iloc[0]
        else:
            df_subject[col_name] = np.nan        
        processed_dfs.append(df_subject)
        
    df = pd.concat(processed_dfs)
    return df

In [149]:
# Add insulin type
df_final = add_single_value_to_subjects(df_final, df_insulin_type, 'insulin_type')
df_final

,CGM,id,bolus,basal,carbs,insulin_type
datetime,,,,,,
2013-09-28 03:35:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart)
2013-09-28 03:40:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart)
2013-09-28 03:45:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart)
2013-09-28 03:50:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart)
2013-09-28 03:55:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart)
...,...,...,...,...,...,...
2015-09-18 12:05:00,106.0,98,NaN,1.00,NaN,Humalog (Lispro)
2015-09-18 12:10:00,104.0,98,NaN,1.00,NaN,Humalog (Lispro)
2015-09-18 12:15:00,100.0,98,NaN,1.00,NaN,Humalog (Lispro)


In [151]:
# Add age
df_age = get_df_from_file(raw_data_file_path, 'HPtRoster.txt', parse_datetime=False, sep='|')
df_age = df_age[['PtID', 'AgeAsOfEnrollDt']]
df_age = df_age.rename(columns={'PtID': 'id', 'AgeAsOfEnrollDt': 'age'})
df_age.head()

,id,age
0,263,44
1,101,27
2,105,42
3,109,78
4,240,68


In [152]:
# Add age
df_final = add_single_value_to_subjects(df_final, df_age, 'age')
df_final.head()

,CGM,id,bolus,basal,carbs,insulin_type,age
datetime,,,,,,,
2013-09-28 03:35:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63
2013-09-28 03:40:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63
2013-09-28 03:45:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63
2013-09-28 03:50:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63
2013-09-28 03:55:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63


In [154]:
# Add gender, weight, height
df_user_data = get_df_from_file(raw_data_file_path, 'HScreening.txt', parse_datetime=False, sep='|')
df_user_data = df_user_data[['PtID', 'Gender', 'Weight', 'Height']]
df_user_data = df_user_data.rename(columns={'PtID': 'id', 'Gender': 'gender', 'Weight': 'weight', 'Height': 'height'})
df_user_data.head()

,id,gender,weight,height
0,40,M,101.3,154.0
1,46,M,107.9,188.9
2,109,F,57.0,163.0
3,70,F,61.0,161.0
4,101,M,75.0,183.0


In [155]:
# Add weight, height, and gender
df_final = add_single_value_to_subjects(df_final, df_user_data, 'weight')
df_final = add_single_value_to_subjects(df_final, df_user_data, 'height')
df_final = add_single_value_to_subjects(df_final, df_user_data, 'gender')
df_final.head()

,CGM,id,bolus,basal,carbs,insulin_type,age,weight,height,gender
datetime,,,,,,,,,,
2013-09-28 03:35:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63,69.0,167.0,F
2013-09-28 03:40:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63,69.0,167.0,F
2013-09-28 03:45:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63,69.0,167.0,F
2013-09-28 03:50:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63,69.0,167.0,F
2013-09-28 03:55:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63,69.0,167.0,F


### Save Resampled Data

In [161]:
df_final = df_final.rename_axis("date")
df_final

,CGM,id,bolus,basal,carbs,insulin_type,age,weight,height,gender
date,,,,,,,,,,
2013-09-28 03:35:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63,69.0,167.0,F
2013-09-28 03:40:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63,69.0,167.0,F
2013-09-28 03:45:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63,69.0,167.0,F
2013-09-28 03:50:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63,69.0,167.0,F
2013-09-28 03:55:00,NaN,10,NaN,0.65,NaN,Novolog (Aspart),63,69.0,167.0,F
...,...,...,...,...,...,...,...,...,...,...
2015-09-18 12:05:00,106.0,98,NaN,1.00,NaN,Humalog (Lispro),62,78.0,176.5,M
2015-09-18 12:10:00,104.0,98,NaN,1.00,NaN,Humalog (Lispro),62,78.0,176.5,M
2015-09-18 12:15:00,100.0,98,NaN,1.00,NaN,Humalog (Lispro),62,78.0,176.5,M


In [159]:
save_file_path = "../../data/resampled/"
os.makedirs(save_file_path, exist_ok=True)
df_final.to_csv(save_file_path + 'ReplaceBG.csv')